In [1]:
from CITNP.datasets.dataset_generator import InterventionDatasetGenerator


datagenerator = InterventionDatasetGenerator(
    function_generator="resnet",
    num_variables=[15],
    sample_size=1000,
    batch_size=16,
    graph_type=["ER"],
    graph_degrees=[15*4],
    iterations_per_epoch=1,
)

In [2]:
import time

# Measure single batch generation time
start_time = time.time()
batch = next(iter(datagenerator))
first_batch_time = time.time() - start_time

# Measure multiple batches
start_time = time.time()
batches = []
for i, batch in enumerate(datagenerator):
    if i >= 100:  # Measure 10 batches
        break
    batches.append(batch)
avg_batch_time = (time.time() - start_time) / 100

print(f"First batch time: {first_batch_time:.4f}s")
print(f"Average batch time: {avg_batch_time:.4f}s")

First batch time: 3.4186s
Average batch time: 0.0331s


Generating resnet data is faster on GPU than on CPU. Howerver, CPU allows for multiple workers.

In [4]:
import time
import torch
from torch.utils.data import DataLoader

def benchmark_dataloader(dataset, num_workers_list, num_batches=20):
    results = {}

    for num_workers in num_workers_list:
        # Create DataLoader with specified workers
        loader = DataLoader(
            dataset=dataset,
            batch_size=None,  # Your generator already creates batches
            num_workers=num_workers,
            pin_memory=True
        )

        # Warmup
        for _ in range(2):
            next(iter(loader))

        # Timing
        start_time = time.time()
        for i, _ in enumerate(loader):
            if i >= num_batches:
                break

        elapsed = time.time() - start_time
        results[num_workers] = elapsed / num_batches

    return results

# Test with different worker counts
dataset = datagenerator
worker_counts = [0, 1, 2, 4, 8, 16]  # Adjust based on your CPU
results = benchmark_dataloader(dataset, worker_counts)

for workers, avg_time in results.items():
    print(f"Workers: {workers}, Avg time per batch: {avg_time:.4f}s")

Workers: 0, Avg time per batch: 3.0612s
Workers: 1, Avg time per batch: 8.6227s
Workers: 2, Avg time per batch: 12.6866s
Workers: 4, Avg time per batch: 12.6301s
Workers: 8, Avg time per batch: 14.5016s
Workers: 16, Avg time per batch: 21.8463s
